# RS-007 학년 가중치 실험 (Grade-Distance Weighting)

RS-006(§9)에서 진단한 문제: `GRADE_TO_BANDS`가 누적 구조라 target_grade 5~6 질의는
grade_band 필터링 효과가 거의 없어 코퍼스 전체(288개)가 무차별 후보가 됨. 이 노트북은
그 진단부터 grade-distance 가중치 프로토타입을 실측으로 검증해 `logic.py`에 반영하기까지의
실험 코드와 결과를 남긴다. 서술형 의사결정 기록은 `RS-006_검색구조_의사결정_기록.md` §9 참고.

**이 노트북에서 재현하는 것**
1. 골든셋(RS-005 최종본, 42행) + 코퍼스(288개 청크) 현황
2. dense 단독 Recall@k (4개 임베딩 모델)
3. 실제 파이프라인: prefilter(hybrid) vs llm_only 비교 (42행)
4. 학년 가중치 스윕 (4세트) → 최종 채택값 도출


## 1. 골든셋 + 코퍼스 현황

In [ ]:
import json
import csv
from collections import Counter
from pathlib import Path

REPO_ROOT = Path("..").resolve()
APP_DATA = REPO_ROOT / "app" / "data"

golden_path = REPO_ROOT / "curriculum-search-engine" / "RS-005_골든셋.csv"
with open(golden_path, encoding="utf-8") as f:
    golden_rows = list(csv.DictReader(f))

corpus = json.loads((APP_DATA / "curriculum_units.json").read_text(encoding="utf-8"))

print(f"골든셋(RS-005_골든셋.csv): {len(golden_rows)}행")
print(f"코퍼스(curriculum_units.json): {len(corpus)}개 청크")
for subject, count in Counter(c["subject"] for c in corpus).items():
    print(f"  {subject}: {count}개")
grade_counts = Counter(r["target_grade"] for r in golden_rows)
print("골든셋 target_grade 분포:", dict(sorted(grade_counts.items())))

골든셋(RS-005_골든셋.csv): 42행
코퍼스(curriculum_units.json): 288개 청크
  MATH: 121개
  SCIENCE: 102개
  DOMESTIC_SCIENCE: 39개
  ART: 26개
골든셋 target_grade 분포: {'2': 2, '3': 4, '4': 8, '5': 9, '6': 19}

## 2. dense 단독 Recall@k (임베딩 모델 4종)

`eval_recall.py`(로컬 캐시, API 불필요)를 그대로 재사용해 4개 임베딩 모델의 순수 코사인
유사도 순위 기준 Recall@k를 비교한다. 참고: 이건 dense 단독 지표라 실제 파이프라인
(dense+sparse RRF+LLM 리랭킹) 성능과는 다를 수 있음(§RS-006 8.8에서 실측으로 확인된 함정).

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "curriculum-search-engine"))
from eval_recall import load_answered_rows, load_chunks, recall_at_k, MODELS, K_VALUES

rows = load_answered_rows()
chunks = load_chunks()
print(f"평가 대상: {len(rows)}개 행\n")

for model_name in MODELS:
    result, _ = recall_at_k(model_name, rows, chunks)
    print(model_name)
    for k in K_VALUES:
        print(f"  Recall@{k}: {result[k]:.2%}")

평가 대상: 42개 행

jhgan/ko-sroberta-multitask
  Recall@1: 11.90%
  Recall@3: 33.33%
  Recall@5: 38.10%
  Recall@10: 52.38%
BAAI/bge-m3
  Recall@1: 21.43%
  Recall@3: 33.33%
  Recall@5: 54.76%
  Recall@10: 64.29%
nlpai-lab/KoE5
  Recall@1: 21.43%
  Recall@3: 42.86%
  Recall@5: 54.76%
  Recall@10: 66.67%
intfloat/multilingual-e5-large
  Recall@1: 28.57%
  Recall@3: 40.48%
  Recall@5: 50.00%
  Recall@10: 66.67%

## 3. 실제 파이프라인: prefilter(hybrid) vs llm_only (42행)

`eval_prefilter_vs_llm_only.py` 실행 결과(`app/data/eval_prefilter_vs_llm_only_results.json`)를
불러온다. hybrid = dense+sparse RRF top-20 → Gemini 리랭킹(현재 프로덕션 구조).
llm_only = grade_band 필터만 거친 후보 전량을 곧장 Gemini에 전달(필터 없음, 대조군).

**목적은 "뭘 배포할지 고르는 것"이 아니라 "prefilter가 정답을 얼마나 걸러내는지" 재는 진단(ablation)이다**
— llm_only는 지연이 SLA(2초)를 사실상 항상 위반해 그대로 배포할 수 없다.

In [ ]:
d = json.loads((APP_DATA / "eval_prefilter_vs_llm_only_results.json").read_text(encoding="utf-8"))
n = len(d)
hybrid_hits = sum(r["hybrid_hit"] for r in d)
llm_hits = sum(r["llm_only_hit"] for r in d)
hybrid_lat = [r["hybrid_elapsed"] for r in d]
llm_lat = [r["llm_only_elapsed"] for r in d]

print(f"골든셋 {n}행\n")
print(f"{'방식':<12}{'Recall':<16}{'평균 지연':<12}{'최대 지연'}")
print(f"{'hybrid':<12}{f'{hybrid_hits}/{n} ({hybrid_hits/n:.2%})':<16}{f'{sum(hybrid_lat)/n:.2f}s':<12}{max(hybrid_lat):.2f}s")
print(f"{'llm_only':<12}{f'{llm_hits}/{n} ({llm_hits/n:.2%})':<16}{f'{sum(llm_lat)/n:.2f}s':<12}{max(llm_lat):.2f}s")

골든셋 42행

방식          Recall          평균 지연       최대 지연
hybrid      20/42 (47.62%)  2.14s       6.96s
llm_only    28/42 (66.67%)  47.59s      101.40s

**행별 대조에서 발견한 것(§RS-006 9.6)**: llm_only만 성공한 케이스의 상당수가
"분류", "비교와 순서화"처럼 **이름이 짧고 범용적인 개념**이었다. 예: 과학 도메인 "분류"(동물,
`4과02-01`)는 dense+sparse가 수학 도메인 "분류"(사각형, `4수03-10`) 쪽으로 끌려가 top-20에서
아예 걸러짐. 이건 학년 거리와는 별개 원인(도메인 모호성)이라, 4절의 학년 가중치만으로는
해결되지 않는다 — 별도 처방이 필요한 남은 문제로 기록해둔다.

## 4. 학년 가중치 스윕 (4세트)

RS-006 §9.7에서 검토한 팀 제안(soft filter/boosting)을 프로토타입: RRF 융합 스코어에
`chunk.grade_band`와 질의의 본 학년군 사이 거리(0/1/2단계)에 따른 가중치를 곱한다
(`logic.py`의 `_grade_band_weight`). "팀 제안값을 그냥 채택"하지 않고, §RS-006 8.7/8.8에서
세운 원칙(LLM-in-the-loop 실측 없이 하이퍼파라미터를 확정하지 않는다)을 그대로 적용해
`sweep_grade_weights.py`로 4개 후보 세트를 동일 조건(42행, pool=20, gemini-flash-lite-latest)에서
비교했다.

In [ ]:
configs = [
    ("baseline_1.0_1.0_1.0", "1.0 / 1.0 / 1.0 (가중치 없음)"),
    ("team_1.0_0.6_0.3", "1.0 / 0.6 / 0.3 (팀 제안값)"),
    ("mild_1.0_0.8_0.6", "1.0 / 0.8 / 0.6 (완만, 채택)"),
    ("strong_1.0_0.4_0.15", "1.0 / 0.4 / 0.15 (강함)"),
]
SWEEP_DIR = APP_DATA / "grade_weight_sweep"

print(f"{'가중치 세트':<32}{'Recall':<16}{'평균 지연':<10}{'최대 지연'}")
for label, name in configs:
    d = json.loads((SWEEP_DIR / f"{label}.json").read_text(encoding="utf-8"))
    n = len(d)
    hits = sum(r["hit"] for r in d)
    lat = [r["elapsed"] for r in d]
    print(f"{name:<32}{f'{hits}/{n} ({hits/n:.2%})':<16}{f'{sum(lat)/n:.2f}s':<10}{max(lat):.2f}s")

가중치 세트                          Recall          평균 지연     최대 지연
1.0 / 1.0 / 1.0 (가중치 없음)        20/42 (47.62%)  1.99s     7.18s
1.0 / 0.6 / 0.3 (팀 제안값)         22/42 (52.38%)  1.98s     4.57s
1.0 / 0.8 / 0.6 (완만, 채택)        27/42 (64.29%)  2.04s     3.65s
1.0 / 0.4 / 0.15 (강함)           24/42 (57.14%)  1.85s     2.60s

## 5. 골든셋 32~43행("2차 추가") 복붙 오류 재검증

pool size가 도움이 될지 보려고 학년 가중치(1.0/0.8/0.6) 적용 후에도 실패한 행들의 RRF 순위를
까보던 중, no.37/38/39/41/42/43의 `성취기준_원문` 컬럼이 서로 다른 `chunk_id`를 가리키면서도
텍스트는 전부 `6실05-01`의 실제 원문이 그대로 복붙되어 있는 것을 발견했다. 32~43행 전체를
코퍼스 원문과 재대조한 결과 4건이 실제로 틀린 코드였다.

In [ ]:
corrections = [
    ("39", "디지털과 아날로그", "6실05-01", "6실05-04", "개념이 05-04 원문(디지털/아날로그 데이터 특징)과 거의 일치"),
    ("41", "기계학습과 딥러닝", "6실05-01", "6실05-05", "05-05(AI가 만들어지는 과정)가 더 직접적으로 연결"),
    ("42", "언플러그드 활동", "6실05-02", "6실05-01", "05-02 원문은 컴퓨터 사용 전제라 '언플러그드' 정의와 모순"),
    ("43", "강화학습", "6실05-03", "6실05-05", "05-03은 협업 코딩 얘기라 강화학습과 무관, 05-05가 적합"),
]
print(f"{'no':<4}{'개념':<18}{'기존 코드':<12}{'정정 코드':<12}{'근거'}")
for no, concept, old, new, reason in corrections:
    print(f"{no:<4}{concept:<18}{old:<12}{new:<12}{reason}")

no  개념                기존 코드     정정 코드     근거
39  디지털과 아날로그       6실05-01    6실05-04    개념이 05-04 원문(디지털/아날로그 데이터 특징)과 거의 일치
41  기계학습과 딥러닝       6실05-01    6실05-05    05-05(AI가 만들어지는 과정)가 더 직접적으로 연결
42  언플러그드 활동        6실05-02    6실05-01    05-02 원문은 컴퓨터 사용 전제라 '언플러그드' 정의와 모순
43  강화학습            6실05-03    6실05-05    05-03은 협업 코딩 얘기라 강화학습과 무관, 05-05가 적합

## 6. 수정된 골든셋 + 채택 가중치(1.0/0.8/0.6)로 재측정

`eval_poolsize_latency.py`(pool=20, `gemini-flash-lite-latest`)를 수정된 골든셋 42행 전체로
재실행한 결과(`app/data/eval_poolsize_latency_results_pool20.json`).

In [ ]:
d = json.loads((APP_DATA / "eval_poolsize_latency_results_pool20.json").read_text(encoding="utf-8"))
n = len(d)
hits = sum(r["hit"] for r in d)
lat = [r["elapsed"] for r in d]
over = sum(1 for t in lat if t > 2.0)
print("모델: gemini-flash-lite-latest, 가중치: 1.0/0.8/0.6, pool=20")
print(f"Recall: {hits}/{n} ({hits/n:.2%}) | 평균 지연: {sum(lat)/n:.2f}s | 최대: {max(lat):.2f}s | 2s SLA 초과: {over}/{n}")

모델: gemini-flash-lite-latest, 가중치: 1.0/0.8/0.6, pool=20
Recall: 27/42 (64.29%) | 평균 지연: 2.07s | 최대: 7.24s | 2s SLA 초과: 18/42

In [ ]:
# 8절(pool size 순위 분석)에서 쓸 RRF 재계산 도구 준비 — 노트북을 처음부터 다시 실행할 때 필요
import numpy as np
sys.path.insert(0, str(REPO_ROOT / "app"))
from agents.curriculum_search.logic import (
    _cosine_similarities, _sparse_scores, _reciprocal_rank_fusion,
    resolve_grade_bands, embedding_source_text, _grade_band_weight,
)
from agents.curriculum_search.schema import CurriculumChunk
from sentence_transformers import SentenceTransformer

chunks_typed = [CurriculumChunk(**c) for c in corpus]
cache = np.load(APP_DATA / "embeddings_cache" / "jhgan__ko-sroberta-multitask.npz")
emb_by_id = dict(zip(cache["chunk_ids"], cache["embeddings"]))
st_model = SentenceTransformer("jhgan/ko-sroberta-multitask")
print("준비 완료:", len(chunks_typed), "개 청크,", len(emb_by_id), "개 임베딩")

준비 완료: 288 개 청크, 288 개 임베딩

## 7. pool size를 늘리면 도움이 될까 — 실패 행의 실제 RRF 순위 분석

pool을 늘리기 전에, 지금 실패하는 15개 행이 애초에 top-20 밖에 있어서 놓치는 건지(retrieval
문제 → pool 확대가 도움) 아니면 top-20 안에 있는데도 LLM이 못 고른 건지(reranking 문제 → pool
확대는 무의미) 먼저 확인한다. `_grade_band_weight`가 반영된 RRF 융합 스코어로 각 실패 행의
정답 순위를 직접 계산.

In [ ]:
results = json.load(open(APP_DATA / "eval_poolsize_latency_results_pool20.json", encoding="utf-8"))
missed_nos = {r["no"] for r in results if not r["hit"]}

within_20, beyond_20 = [], []
for i, row in enumerate(golden_rows, start=2):
    no = str(i)
    if no not in missed_nos:
        continue
    grade = int(row["target_grade"])
    gold_ids = {g.strip() for g in row["chunk_id"].split(",")}
    bands = {b.value for b in resolve_grade_bands(grade)}
    cand = [c for c in chunks_typed if c.grade_band.value in bands]
    embeddings = [emb_by_id[c.chunk_id] for c in cand]
    q_emb = st_model.encode(row["개념_정의_초안"]).tolist()
    dense = _cosine_similarities(q_emb, embeddings)
    texts = [embedding_source_text(c) for c in cand]
    sparse = _sparse_scores(f"{row['ai_개념']} {row['개념_정의_초안']}", texts)
    fused = _reciprocal_rank_fusion(dense, sparse)
    query_band = resolve_grade_bands(grade)[-1]
    weighted = [f * _grade_band_weight(c.grade_band, query_band) for c, f in zip(cand, fused)]
    order = sorted(range(len(cand)), key=lambda i: weighted[i], reverse=True)
    rank = next((pos + 1 for pos, idx in enumerate(order) if cand[idx].chunk_id in gold_ids), None)
    (within_20 if rank and rank <= 20 else beyond_20).append(no)
    print(f"no.{no:<3} {row['ai_개념']:<20} 순위:{rank}")

print(f"\n총 미스 {len(missed_nos)}개 중 within-20(pool 늘려도 소용없음): {len(within_20)}개, beyond-20(pool 늘리면 구제 가능): {len(beyond_20)}개")

no.4   패턴 인식                순위:1
no.5   예측                   순위:9
no.6   특징 추출                순위:10
no.11  데이터 시각화              순위:1
no.13  반복과 규칙 적용            순위:7
no.14  비교와 순서화              순위:30
no.20  분류 기준 설계             순위:26
no.21  의사결정트리               순위:5
no.22  분수 기반 픽셀 표현          순위:2
no.24  그림그래프 기반 데이터 시각화     순위:13
no.28  분류                   순위:11
no.29  데이터의 시각화             순위:47
no.32  공통점과 차이점 구별하기        순위:9
no.38  자연어                  순위:12
no.39  디지털과 아날로그            순위:1

총 미스 15개 중 within-20(pool 늘려도 소용없음): 12개, beyond-20(pool 늘리면 구제 가능): 3개

**결론**: 15개 실패 중 12개(80%)는 정답이 이미 top-20 안(심지어 no.4/no.11/no.39는 1위)에
있었는데도 LLM이 못 골랐다. pool size 확대로 구제 가능한 건 3개(no.14/20/29)뿐이라 최선의
경우도 +7.1%p — pool size는 지금 우선순위가 아니라고 판단, 보류.

## 8. 결정적 발견 — 평가에 쓰던 모델(lite) 자체가 저평가 원인

"정답이 1위인데도 왜 거부됐는지" 확인하려고 no.4/no.11/no.39를 `search_within_chunks`로 직접
재현하다가, 지금까지 모든 eval이 무료 티어 쿼터 문제로 프로덕션 모델(`gemini-flash-latest`)
대신 `gemini-flash-lite-latest`를 써왔다는 게 실제로 recall에 얼마나 영향을 주는지 직접
비교했다.

**행별 비교** (동일 후보 목록, 모델만 다름):

| 케이스 | lite 모델 | flash-latest(프로덕션) |
| --- | --- | --- |
| no.4 패턴 인식 | 빈 리스트(실패) | 정답 1위로 정확히 선택 |
| no.11 데이터 시각화 | 오답 1위, 정답 2위 | 정답 1위로 정확히 선택 |
| no.39 디지털과 아날로그 | 빈 리스트(실패) | 정답 1위로 정확히 선택 |

**42행 전체 재측정** (`RERANK_MODEL=gemini-flash-latest`로 `eval_poolsize_latency.py` 재실행)

In [ ]:
lite = json.loads((APP_DATA / "eval_poolsize_latency_results_pool20.json").read_text(encoding="utf-8"))
flash = json.loads((APP_DATA / "eval_poolsize_latency_results_pool20_flashlatest.json").read_text(encoding="utf-8"))

for label, d in [("gemini-flash-lite-latest (eval용)", lite), ("gemini-flash-latest (프로덕션)", flash)]:
    n = len(d)
    hits = sum(r["hit"] for r in d)
    lat = [r["elapsed"] for r in d]
    over = sum(1 for t in lat if t > 2.0)
    print(f"{label:<32} Recall={hits}/{n}({hits/n:.2%})  평균지연={sum(lat)/n:.2f}s  최대={max(lat):.2f}s  SLA초과={over}/{n}")

gemini-flash-lite-latest (eval용) Recall=27/42(64.29%)  평균지연=2.07s  최대=7.24s  SLA초과=18/42
gemini-flash-latest (프로덕션)       Recall=35/42(83.33%)  평균지연=6.59s  최대=11.50s  SLA초과=42/42

**REQ003 목표(80%)를 최초로 넘김(83.33%).** 하지만 지연이 3배 이상 늘어(2.07s → 6.59s) 2초
SLA를 42행 전부 위반하는 새로운 문제가 생겼다 — recall과 latency가 정면으로 충돌하는
트레이드오프가 확인됨. "리랭커/프롬프트를 바꿔야 한다"는 가설은 기각 — 실제 프로덕션
모델은 처음부터 잘 작동했고, 문제는 **평가에 쓰던 모델이 프로덕션 모델보다 약했다는 것**이었다.

**⚠️ 무효화됨(§9 참고)**: 이 83.33%/6.59s는 잘못된 모델(`gemini-flash-latest` alias, 구 SDK 직접 호출)로 측정된 값. 정정된 구성(`gemini-3.6-flash` + `lib.gemini.generate_structured()`)의 실측치는 §10 참고.

## 9. 정정 — 프로덕션 모델은 `gemini-flash-latest`가 아니라 `gemini-3.6-flash` + 팀 공용 래퍼

§8의 "결정적 발견"은 `logic.py`가 `google.generativeai`(구 SDK)로 `gemini-flash-latest`(alias)를
직접 호출하던 상태에서 측정한 것이었다. 이후 팀 리뷰로 다음이 확인됨:

1. **모델**: 팀 표준은 alias가 아니라 버전 고정 문자열 `gemini-3.6-flash`(NFR-001-6 재현성 요구).
2. **호출 경로**: 전 에이전트는 `app/lib/gemini.py`의 `generate_structured()`를 경유해야 한다
   (팀 규약 — `google.genai` 직접 호출 금지). §8까지는 이 규약을 위반한 상태였다.
3. **thinking 제어**: `gemini-3.6-flash`부터 `temperature`/`top_p`/`top_k`가 deprecated되고,
   `thinking_level`(`minimal`/`low`/`medium`/`high`)로 추론량을 조절한다. §8에서 실험했던
   `thinking_budget`(정수 토큰 예산)은 구세대 파라미터라 애초에 안 맞는 걸 실측했던 셈이다.

`logic.py`를 `lib.gemini.generate_structured()` 경유로 교체(`RERANK_MODEL="gemini-3.6-flash"`,
`RERANK_THINKING_LEVEL="low"`)하고 실제 검색 스모크 테스트로 정상 동작을 확인한 뒤,
아래 §10에서 `thinking_level` 3점을 다시 실측했다. **§8의 83.33%/6.59s 수치는 무효.**
(상세: `RS-006_검색구조_의사결정_기록.md` §9.12)

## 10. `thinking_level` 3점(minimal/low/medium) 재실측

정정된 구성(`gemini-3.6-flash`, `lib.gemini.generate_structured()` 경유)으로
`eval_poolsize_latency.py`를 `thinking_level` minimal/low/medium 각각에 대해 42행 전량 재실행.

In [ ]:
for label in ["minimal", "low", "medium"]:
    d = json.loads((APP_DATA / f"eval_poolsize_latency_results_pool20_gemini36flash_{label}.json").read_text(encoding="utf-8"))
    n = len(d)
    hits = sum(r["hit"] for r in d)
    lat = [r["elapsed"] for r in d]
    over = sum(1 for t in lat if t > 2.0)
    print(f"{label:<8} Recall={hits}/{n}({hits/n:.2%})  평균지연={sum(lat)/n:.2f}s  최대={max(lat):.2f}s  SLA초과={over}/{n}")

minimal  Recall=34/42(80.95%)  평균지연=3.19s  최대=9.64s  SLA초과=41/42
low      Recall=33/42(78.57%)  평균지연=3.07s  최대=8.78s  SLA초과=40/42
medium   Recall=35/42(83.33%)  평균지연=6.48s  최대=12.01s  SLA초과=42/42

**minimal과 low는 recall(78.57~80.95%)·지연(3.07~3.19s) 둘 다 사실상 동일** — 1행 차이는 LLM
비결정성 노이즈로 해석(이 모델은 재현성이 완전 보장되지 않음).   
 **medium만 확실히 다른 지점**:
recall +2.4~4.76%p를 얻는 대가로 지연이 2배 이상(6.48s, 최대 12.01s)이 된다.

**결정 — `RERANK_THINKING_LEVEL = "low"` 유지, medium 채택하지 않음.**
1. medium의 recall 이득(42개 중 1~2행)은 B(매핑 에이전트)의 B-06(fallback)·B-07(재매핑 요청)로
   파이프라인 차원에서 일부 흡수됨 — A2 단독 recall 갭이 그대로 시스템 실패로 이어지지 않는다.
2. 남은 recall 갭(§3에서 찾은 도메인 모호성 문제 등)은 latency 비용 없이 메울 여지가 있다.
3. medium의 최대 지연(12.01s)은 사용자 체감상 "멈춘 것처럼" 보일 위험이 크다.

**NFR-002-1 재조정 제안**: 2초 → 3~3.5초(팀 승인 대기, `docs/curriculum_search/REQ002-교육과정검색엔진.md` v3.2 참고).

**⚠️ 위 수치 무효화됨(§10.1 참고, 2026-08-06)**: 이 스윕은 `eval_poolsize_latency.py`의
`EMBEDDING_CACHE`가 `jhgan__ko-sroberta-multitask.npz`로 하드코딩된 상태에서 돈 것이었다 —
§11에서 나중에 KoE5로 교체하기로 결정했지만, 이 §10 스윕 자체는 그 교체 전 구성(ko-sroberta)
그대로 남아 있었다(수정 안 함). 실제 프로덕션 임베딩(KoE5)으로 다시 돌리면 결론이 바뀐다:
아래 §10.1 참고.

## 11. 결론

- **학년 가중치**: `_GRADE_DISTANCE_WEIGHT = {0: 1.0, 1: 0.8, 2: 0.6}` 채택 — 비단조 관계(페널티가
  강할수록 좋아지는 게 아니라 완만한 값이 최적)를 실측으로 확인.
- **골든셋 품질**: 32~43행("2차 추가")에서 복붙으로 인한 코드 오류 4건 발견·수정.
- **pool size**: 실패 행 대부분(80%)이 이미 top-20 안에 정답이 있던 경우라 낮은 우선순위로 판단.
- **모델/SDK 정정**: 프로덕션 모델은 `gemini-flash-latest`(alias, 구 SDK 직접 호출)가 아니라
  `gemini-3.6-flash` + `app/lib/gemini.py` 공용 래퍼가 팀 규약상 맞는 구성. §8의 83.33%/6.59s는
  이 정정 전에 잘못된 모델/파라미터로 측정된 무효 수치.
- **`thinking_level` 채택**: `"low"` — **(2026-08-06 갱신, §10.1)** KoE5 임베딩 + 프로덕션
  리랭커(`gemini-3.6-flash`)로 재검증한 결과 Recall 85.71%(36/42, minimal과 동률 1위)·평균
  지연 3.15s로 **모든 지표에서 medium(83.33%/6.39s)보다 우위**. §10(ko-sroberta 기준)에서는
  medium이 recall 1등이라 "지연 비용이 과해 기각"이라는 소거법으로 결정했었으나, KoE5로 바꾸며
  전제 자체가 바뀌어 low 채택 근거가 트레이드오프 판단에서 직접 우위로 강화됨.
- **(갱신) NFR-002-1 팀 승인 완료(2026-08-05)** — 2초 → 3~3.5초로 확정 (`REQ002` 문서 반영).
- **(갱신) 도메인 모호성 문제 재확인 → 처방 불필요로 결론.** §3에서 찾은 "분류"(수학 vs 과학)
  혼동은 정정된 모델(`gemini-3.6-flash`)로 재실행하니 재현되지 않음 — lite 모델(당시 eval 대체
  모델)의 약점이었을 뿐 실제 프로덕션 구성의 문제가 아니었음. 상세: `RS-006` §9.17.
- **(갱신) 임베딩 모델 재검증 → KoE5로 교체.** 정정된 리랭커로 4개 모델을 다시 end-to-end
  비교한 결과 `ko-sroberta-multitask`(78.57%) → `nlpai-lab/KoE5`(83.33%, 재현 시 78.57~83.33%
  변동)로 순위가 뒤집혀 `EMBEDDING_MODEL` 교체. 상세·재현성 노이즈 논의: `RS-006` §9.18,
  `RS-003_임베딩모델_벤치마킹.ipynb`.
- **(신규, 2026-08-06) NFR-002-2 프로덕션 실측치 갱신**: 기존 83.33%(35/42)는
  `eval_embedding_e2e.py`가 쿼터 제약으로 대체 리랭커(`gemini-flash-lite-latest`)를 썼던 값.
  실제 프로덕션 리랭커(`gemini-3.6-flash`, `thinking_level="low"`)로 §10.1에서 재측정한
  **85.71%(36/42)**가 더 정확한 수치 — `REQ002` NFR-002-2 갱신 반영.
- **다음 과제**: 현재 없음 — 이번 세션에서 식별된 항목은 전부 처리 완료. Cloud SQL 인스턴스
  생성 후 실 데이터 적재·`hybrid_search()`(로컬 캐시 아닌 실 DB 경로) 스모크 테스트만 남음.


In [1]:
for label in ["minimal", "low", "medium"]:
    d = json.loads((APP_DATA / f"eval_poolsize_latency_results_pool20_koe5_{label}.json").read_text(encoding="utf-8"))
    n = len(d)
    hits = sum(r["hit"] for r in d)
    lat = [r["elapsed"] for r in d]
    over = sum(1 for t in lat if t > 2.0)
    print(f"{label:<8} Recall={hits}/{n}({hits/n:.2%})  평균지연={sum(lat)/n:.2f}s  최대={max(lat):.2f}s  SLA초과={over}/{n}")

minimal  Recall=36/42(85.71%)  평균지연=3.14s  최대=12.98s  SLA초과=40/42
low      Recall=36/42(85.71%)  평균지연=3.15s  최대=13.06s  SLA초과=39/42
medium   Recall=35/42(83.33%)  평균지연=6.39s  최대=18.72s  SLA초과=42/42


**§10과 순위가 달라졌다.** ko-sroberta 기준으로는 medium이 recall 1등(83.33%)이었는데, KoE5로
바꾸니 **low가 minimal과 동률로 1등(85.71%)이고 medium은 오히려 더 낮다(83.33%)** — recall과
지연이 트레이드오프 관계라는 §10의 전제 자체가 더는 성립하지 않는다. 지금은 low가 recall도
가장 좋고 지연도 가장 짧은 완전 우위 상태다.

**결정 — `RERANK_THINKING_LEVEL = "low"` 유지(재확인, 근거 강화).** §10의 "recall 이득 대비
지연 손해가 커서 medium을 안 쓴다"는 소거법 논리에서, "low가 모든 지표에서 medium보다 낫다"는
직접적 근거로 바뀌었다. minimal과 recall이 동률인 이유로 low를 유지하는 근거(예비 마진 —
API 쪽 thinking_level 기본 거동 변화 등에 대비)는 §10과 동일하게 유효.

**NFR-002-2 갱신 필요**: `REQ002`의 프로덕션 실측치(83.33%, `eval_embedding_e2e.py`)는
쿼터 제약으로 대체 모델(`gemini-flash-lite-latest`)을 쓴 값이었다 — 실제 프로덕션 리랭커
(`gemini-3.6-flash`)로 재보니 **85.71%(36/42)**로 더 높다. `REQ002` NFR-002-2 갱신 반영.

## 11. 결론

- **학년 가중치**: `_GRADE_DISTANCE_WEIGHT = {0: 1.0, 1: 0.8, 2: 0.6}` 채택 — 비단조 관계(페널티가
  강할수록 좋아지는 게 아니라 완만한 값이 최적)를 실측으로 확인.
- **골든셋 품질**: 32~43행("2차 추가")에서 복붙으로 인한 코드 오류 4건 발견·수정.
- **pool size**: 실패 행 대부분(80%)이 이미 top-20 안에 정답이 있던 경우라 낮은 우선순위로 판단.
- **모델/SDK 정정**: 프로덕션 모델은 `gemini-flash-latest`(alias, 구 SDK 직접 호출)가 아니라
  `gemini-3.6-flash` + `app/lib/gemini.py` 공용 래퍼가 팀 규약상 맞는 구성. §8의 83.33%/6.59s는
  이 정정 전에 잘못된 모델/파라미터로 측정된 무효 수치.
- **`thinking_level` 채택**: `"low"` — Recall 78.57~80.95%(재현 시 소폭 변동), 평균 지연 ~3.1s.
  medium(83.33%/6.48s)은 recall은 더 좋지만 지연 비용이 과해 기각.
- **(갱신) NFR-002-1 팀 승인 완료(2026-08-05)** — 2초 → 3~3.5초로 확정 (`REQ002` 문서 반영).
- **(갱신) 도메인 모호성 문제 재확인 → 처방 불필요로 결론.** §3에서 찾은 "분류"(수학 vs 과학)
  혼동은 정정된 모델(`gemini-3.6-flash`)로 재실행하니 재현되지 않음 — lite 모델(당시 eval 대체
  모델)의 약점이었을 뿐 실제 프로덕션 구성의 문제가 아니었음. 상세: `RS-006` §9.17.
- **(갱신) 임베딩 모델 재검증 → KoE5로 교체.** 정정된 리랭커로 4개 모델을 다시 end-to-end
  비교한 결과 `ko-sroberta-multitask`(78.57%) → `nlpai-lab/KoE5`(83.33%, 재현 시 78.57~83.33%
  변동)로 순위가 뒤집혀 `EMBEDDING_MODEL` 교체. 상세·재현성 노이즈 논의: `RS-006` §9.18,
  `RS-003_임베딩모델_벤치마킹.ipynb`.
- **다음 과제**: 현재 없음 — 이번 세션에서 식별된 항목은 전부 처리 완료. Cloud SQL 인스턴스
  생성 후 실 데이터 적재·`hybrid_search()`(로컬 캐시 아닌 실 DB 경로) 스모크 테스트만 남음.
